# 06 — Iceberg medallion: same data, REST catalog

Rebuilds the medallion lakehouse on top of the **Iceberg REST catalog** (`iceberg` namespace `demo`). Same source data as notebook 05.

Highlights specific to Iceberg:
- Hidden partitioning via `days(ts)` — no extra columns on the producer side.
- Metadata tables (`.snapshots`, `.files`, `.history`) for audit.
- Snapshot-id time travel (cleaner than timestamp-based when you have a specific snapshot pinned).

In [ ]:
from spark_session import get_spark
from transforms import explode_pageviews
from pyspark.sql import functions as F

spark = get_spark("06-iceberg-medallion")

# Reset the schema namespaces so re-runs are clean.
for tbl in ("bronze_movies", "bronze_ratings", "bronze_tags", "bronze_pageviews",
            "silver_movies", "silver_ratings", "silver_pageviews",
            "gold_movie_kpis", "gold_genre_kpis", "gold_daily_top"):
    spark.sql(f"DROP TABLE IF EXISTS iceberg.demo.{tbl}")
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.demo")

## Bronze — load raw → Iceberg, no schema changes

In [ ]:
RAW = "s3a://raw-data"

for name in ("movies", "ratings", "tags"):
    (spark.read.option("header", True).csv(f"{RAW}/movielens/{name}.csv")
         .writeTo(f"iceberg.demo.bronze_{name}").createOrReplace())
    print(f"  bronze_{name} ok")

(spark.read.json(f"{RAW}/wikipedia/*.json")
      .writeTo("iceberg.demo.bronze_pageviews").createOrReplace())
print("  bronze_pageviews ok")

## Silver — typed schema; `silver_pageviews` partitioned by day

In [ ]:
spark.sql("""
CREATE TABLE iceberg.demo.silver_movies (
    movie_id     BIGINT,
    title_clean  STRING,
    year         INT,
    genres       ARRAY<STRING>
) USING iceberg
""")
(spark.table("iceberg.demo.bronze_movies")
    .select(
        F.col("movieId").cast("long").alias("movie_id"),
        F.regexp_replace("title", r"\s*\(\d{4}\)$", "").alias("title_clean"),
        F.regexp_extract("title", r"\((\d{4})\)$", 1).cast("int").alias("year"),
        F.split("genres", r"\|").alias("genres"),
    )
    .where("movie_id IS NOT NULL")
    .writeTo("iceberg.demo.silver_movies").append())

spark.sql("""
CREATE TABLE iceberg.demo.silver_ratings (
    user_id   BIGINT,
    movie_id  BIGINT,
    rating    DOUBLE,
    rated_at  TIMESTAMP
) USING iceberg
""")
(spark.table("iceberg.demo.bronze_ratings")
    .select(
        F.col("userId").cast("long").alias("user_id"),
        F.col("movieId").cast("long").alias("movie_id"),
        F.col("rating").cast("double").alias("rating"),
        F.from_unixtime(F.col("timestamp").cast("long")).cast("timestamp").alias("rated_at"),
    )
    .where("user_id IS NOT NULL AND movie_id IS NOT NULL")
    .writeTo("iceberg.demo.silver_ratings").append())

# Wikipedia silver: explode (shared with notebook 05) + partition by day
spark.sql("""
CREATE TABLE iceberg.demo.silver_pageviews (
    day      DATE,
    project  STRING,
    article  STRING,
    views    BIGINT,
    rank     INT
) USING iceberg
PARTITIONED BY (day)
""")
pv = explode_pageviews(spark.table("iceberg.demo.bronze_pageviews"))
pv.writeTo("iceberg.demo.silver_pageviews").append()

print("silver counts:")
for t in ("silver_movies", "silver_ratings", "silver_pageviews"):
    print(f"  iceberg.demo.{t:<18} {spark.table('iceberg.demo.' + t).count():>7,}")

## Gold — same KPIs, written via SQL CTAS

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE iceberg.demo.gold_movie_kpis USING iceberg AS
SELECT
    r.movie_id,
    m.title_clean,
    m.year,
    m.genres,
    COUNT(*)                AS n_ratings,
    ROUND(AVG(r.rating), 3) AS avg_rating
FROM iceberg.demo.silver_ratings r
JOIN iceberg.demo.silver_movies  m USING (movie_id)
GROUP BY r.movie_id, m.title_clean, m.year, m.genres
HAVING COUNT(*) >= 50
""")
spark.sql("""
SELECT title_clean, year, n_ratings, avg_rating
FROM iceberg.demo.gold_movie_kpis
ORDER BY avg_rating DESC, n_ratings DESC
LIMIT 20
""").show(truncate=False)

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE iceberg.demo.gold_genre_kpis USING iceberg AS
SELECT
    genre,
    COUNT(*)                  AS n_movies,
    ROUND(AVG(avg_rating), 3) AS genre_avg_rating,
    SUM(n_ratings)            AS total_ratings
FROM (
    SELECT EXPLODE(genres) AS genre, avg_rating, n_ratings
    FROM iceberg.demo.gold_movie_kpis
)
GROUP BY genre
ORDER BY genre_avg_rating DESC
""")
spark.table("iceberg.demo.gold_genre_kpis").show(truncate=False)

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE iceberg.demo.gold_daily_top USING iceberg AS
SELECT day, article, views
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY day ORDER BY views DESC) AS rn
    FROM iceberg.demo.silver_pageviews
) WHERE rn = 1
ORDER BY day
""")
spark.table("iceberg.demo.gold_daily_top").show(truncate=False)

## Metadata tables — snapshots, files, history

Every Iceberg table exposes `.snapshots`, `.files`, `.history`, `.manifests`, `.partitions`.

In [ ]:
spark.sql("SELECT committed_at, snapshot_id, operation, summary['added-records'] AS added_records FROM iceberg.demo.silver_ratings.snapshots").show(truncate=False)
spark.sql("SELECT partition, record_count, file_size_in_bytes FROM iceberg.demo.silver_pageviews.partitions ORDER BY record_count DESC LIMIT 5").show(truncate=False)

## Snapshot-id time travel

Append a tiny correction, then read the *previous* snapshot to prove time travel works.

In [ ]:
# Snapshot 1 — current
snap_before = spark.sql("SELECT snapshot_id FROM iceberg.demo.silver_ratings.snapshots ORDER BY committed_at DESC LIMIT 1").collect()[0][0]
rows_before = spark.table("iceberg.demo.silver_ratings").count()
print(f"before: snapshot={snap_before}  rows={rows_before:,}")

# Add 3 synthetic ratings
from datetime import datetime
extra = spark.createDataFrame(
    [(99999, 1, 5.0, datetime(2026, 6, 3, 12, 0)),
     (99999, 2, 4.5, datetime(2026, 6, 3, 12, 5)),
     (99999, 3, 5.0, datetime(2026, 6, 3, 12, 10))],
    "user_id LONG, movie_id LONG, rating DOUBLE, rated_at TIMESTAMP")
extra.writeTo("iceberg.demo.silver_ratings").append()

snap_after = spark.sql("SELECT snapshot_id FROM iceberg.demo.silver_ratings.snapshots ORDER BY committed_at DESC LIMIT 1").collect()[0][0]
rows_after = spark.table("iceberg.demo.silver_ratings").count()
print(f"after : snapshot={snap_after}  rows={rows_after:,}")

# Time travel to the previous snapshot — row count should match `rows_before`
rows_travel = spark.read.option("snapshot-id", snap_before).table("iceberg.demo.silver_ratings").count()
print(f"travel→ snapshot={snap_before}  rows={rows_travel:,}")
assert rows_travel == rows_before, "time travel should restore pre-append count"

In [ ]:
# Drop the SparkSession so the History Server can ingest this app's event log
spark.stop()